In [1]:
import pandas as pd

In [ ]:
def calc_zscore_and_remove(
    df: pd.DataFrame,
    columns: list[str],
    threshold: float = 3.0,
) -> pd.DataFrame:
    """Return a copy of df without rows containing z-score outliers."""
    if threshold <= 0:
        raise ValueError("The threshold must be equal to or greater than zero")
    selected = df.loc[:, columns]
    z_scores = (selected - selected.mean()) / (selected.std(ddof=0) + 1e-5)
    outlier_rows = z_scores.abs().gt(threshold).any(axis=1)
    non_outliers = df.loc[~outlier_rows].copy()
    return non_outliers

In [2]:
df = pd.read_csv('heart.csv')
print(df.columns)
df.head()

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='str')


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
"""Preprocessing utilities for tabular ML pipelines: categorical encoding and train/test data loading."""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


def encoding(df: pd.DataFrame) -> pd.DataFrame:
    """Convert categorical (object/category) columns into one-hot encoded columns.

    Args:
        df: Input DataFrame containing categorical feature columns to encode.
            Should NOT include the target column, to keep the label out of
            the feature space.

    Returns:
        A new DataFrame with categorical columns replaced by 0/1 dummy
        columns. The first category of each column is dropped
        (drop_first=True) to avoid the dummy variable trap.

    Note:
        pandas.get_dummies is stateless: it only looks at categories
        present in `df` at call time. If train and test sets are encoded
        *separately*, a category missing from one split will produce
        mismatched columns between them. Safer alternatives for a
        production pipeline: encode the full dataset once before
        splitting, or use
        sklearn.preprocessing.OneHotEncoder(handle_unknown="ignore")
        fit only on the training data so unseen test-time categories
        are handled explicitly instead of silently misaligning columns.
    """
    return pd.get_dummies(df, drop_first=True, dtype=int)


def dataloader(
    X: pd.DataFrame,
    y: np.ndarray,
    test_size: float = 0.2,
    random_state: int = 42,
    stratify_target: bool = False,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Split features/target into train/test sets and scale the features.

    Args:
        X: Feature DataFrame (already encoded — see `encoding()`).
        y: Target array. Returned unscaled: scaling labels is invalid for
            classification and unnecessary (and metric-obscuring) for
            regression unless explicitly inverse-transformed later.
        test_size: Fraction of rows reserved for the test set.
        random_state: Seed for a reproducible split.
        stratify_target: If True, preserves the class distribution of `y`
            across the train/test split — recommended for imbalanced
            classification targets. Ignored for regression
            targets.

    Returns:
        Tuple of (X_train_scaled, X_test_scaled, y_train, y_test) as
        numpy arrays. Features are scaled with a StandardScaler fit
        *only* on the training set, then reused (never refit) to
        transform the test set — this avoids data leakage from the test distribution.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify_target else None,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)  # learn mean/std from train only
    X_test_scaled = scaler.transform(X_test)         # reuse train statistics (typo `transfom` fixed)

    return X_train_scaled, X_test_scaled, y_train, y_test

In [4]:
"""Ensemble model comparison utilities: train and evaluate Bagging, Boosting, and Stacking classifiers."""

import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report


def ensembles(X: pd.DataFrame, y: np.ndarray, random_state: int = 42) -> dict[str, dict[str, float]]:
    """Train Bagging, Boosting, and Stacking classifiers and compare their test-set metrics.

    Args:
        X: Feature DataFrame (already encoded/scaled — see `encoding()` and
            `dataloader()`).
        y: Target array (class labels).
        random_state: Seed passed to every model and to the train/test
            split (via `dataloader`) for reproducible comparisons across
            runs. Without a fixed seed, apparent performance
            differences between models could just be split/initialization
            noise rather than real differences.

    Returns:
        A dict keyed by model name ("bagging", "boosting", "stacking"),
        each mapping to a dict of weighted-average metrics:
        {"accuracy": float, "f1": float, "precision": float, "recall": float}.
        Weighted averages account for class imbalance by weighting
        each class's contribution by its support (true sample count).

    Example:
        >>> results = ensembles(X, y, random_state=0)
        >>> results["stacking"]["f1"]
        0.87
    """
    X_train, X_test, y_train, y_test = dataloader(X, y, random_state=random_state)

    bagging = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=random_state),
        n_estimators=100,
        max_samples=0.8,
        random_state=random_state,
    )
    boosting = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=random_state),
        n_estimators=100,
        random_state=random_state,
    )
    stacking = StackingClassifier(
        estimators=[
            ("lr", LogisticRegression()),
            ("svc", SVC(probability=True)),
            ("tree", DecisionTreeClassifier(random_state=random_state)),
        ],
        final_estimator=LogisticRegression(),
    )

    models = {"bagging": bagging, "boosting": boosting, "stacking": stacking}
    results: dict[str, dict[str, float]] = {}

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # y_true first, y_pred second — reversing these mislabels ground truth
        report = classification_report(y_test, y_pred, output_dict=True)
        weighted = report["weighted avg"]  # accounts for class imbalance

        results[model_name] = {
            "accuracy": report["accuracy"],
            "f1": weighted["f1-score"],
            "precision": weighted["precision"],
            "recall": weighted["recall"],
        }

    return results

In [5]:
df_encoded = encoding(df)
X = df_encoded.drop(columns=['HeartDisease'])
y = df_encoded['HeartDisease'].copy()
results = ensembles(X, y)
print(results)

d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was dep

{'bagging': {'accuracy': 0.8532608695652174, 'f1': 0.8536196666058554, 'precision': 0.8544366638795985, 'recall': 0.8532608695652174}, 'boosting': {'accuracy': 0.8532608695652174, 'f1': 0.8539697542533081, 'precision': 0.8571557971014493, 'recall': 0.8532608695652174}, 'stacking': {'accuracy': 0.875, 'f1': 0.8753056419235065, 'precision': 0.8760817307692308, 'recall': 0.875}}


d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Training\data-science-roadmap\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
